In [8]:
import time
import numpy as np
import pandas as pd
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

def f1m(a, b):
    return f1_score(a, b, average="macro", labels=[0,1], zero_division=0)

def dac_trung(s):
    t = s.split()
    d = [len(x) for x in t]
    return [len(t), len(s), float(np.mean(d)), max(d), min(d),
            sum(1 for x in t if "-" in x), len(set(t)), s.count("-")]

def chay_bang(ten_bang, mo_hinh_list, Xh, Xk, yh, yk):
    print(ten_bang)
    print(f"{'Mo hinh':<30}{'Tap da hoc':>12}{'kiem dinh':.12}{'Giay':>8}")
    for ten, m in mo_hinh_list:
        t0 = time.perf_counter()
        m.fit(Xh, yh)
        f1_hoc = f1_score(yh, m.predict(Xh))
        f1_kiem = f1_score(yk, m.predict(Xk))
        giay = time.perf_counter() - t0
        print(f"{ten:<28}{f1_hoc:>12.4f}{f1_kiem:>12.4f}{giay:>8.1f}")


In [9]:
cot = ["nguon", "nhan", "nhan_goc", "cau"]
df = pd.read_csv("in_domain_train.tsv", sep="\t",
                  header=None, names=cot, quoting=3)

cau = df["cau"].astype(str).values
nhan = df["nhan"].values

In [11]:
X = np.array([dac_trung(s) for s in df["cau"]], dtype=float)

Xh, Xk, yh, yk = train_test_split(X, nhan, test_size=0.2, random_state=0)

scaler = StandardScaler().fit(Xh)
Xh_chuan = scaler.transform(Xh)
Xk_chuan = scaler.transform(Xk)

BANG_A =[
    ("KNN k = 5, khong chuan hoa", KNeighborsClassifier(n_neighbors=5)),
    ("KNN k = 5, co chuan hoa", KNeighborsClassifier(n_neighbors=5)),
    ("cay, khong gioi han", DecisionTreeClassifier(random_state=0)),
    ("cay, do sau 5", DecisionTreeClassifier(max_depth=5, random_state=0)),
    ("rung 300", RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=1)),
    ("boosting", HistGradientBoostingClassifier(random_state=0)),
    ("hoi quy logistic", LogisticRegression(max_iter=2000, random_state=0)),
]
print("\nBang A: 8 dac trung tay")
print(f"{'Mo hinh':<28}{'Tap da hoc':>12}{'Kiem dinh':>12}{'Giay':>8}")
for i, (ten,m) in enumerate(BANG_A):
    t0 = time.perf_counter()
    if i==0: # khong chuan hoa
        m.fit(Xh, yh)
        f1_hoc, f1_kiem = f1m(yh, m.predict(Xh)), f1m(yk, m.predict(Xk))
    elif i==1: #chuan hoa
        m.fit(Xh_chuan, yh)
        f1_hoc, f1_kiem = f1m(yh, m.predict(Xh_chuan)), f1m(yk, m.predict(Xk_chuan))
    else:
        m.fit(Xh, yh)
        f1_hoc, f1_kiem = f1m(yh, m.predict(Xh)), f1m(yk, m.predict(Xk))
    giay = time.perf_counter() - t0
    print(f"{ten:<28}{f1_hoc:>12.4f}{f1_kiem:>12.4f}{giay:>8.1f}")


Bang A: 8 dac trung tay
Mo hinh                       Tap da hoc   Kiem dinh    Giay
KNN k = 5, khong chuan hoa        0.6183      0.4970     0.0
KNN k = 5, co chuan hoa           0.6165      0.4900     0.1
cay, khong gioi han               0.7701      0.5238     0.0
cay, do sau 5                     0.4456      0.4186     0.0
rung 300                          0.7453      0.5199     1.7
boosting                          0.4971      0.4318     2.0
hoi quy logistic                  0.4134      0.4126     0.0


In [12]:
tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2,4), max_features=2143)
X_tfidf = tfidf.fit_transform(cau).toarray()

Xh_t, Xk_t, yh_t, yk_t = train_test_split(X_tfidf, nhan, test_size=0.2, random_state=0)

BANG_B = [
    ("KNN k=5", KNeighborsClassifier(n_neighbors=5)),
    ("cay quyet dinh", DecisionTreeClassifier(random_state=0)),
    ("rung 300 cay", RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=1)),
    ("boosting", HistGradientBoostingClassifier(random_state=0)),
    ("hoi quy logistic", LogisticRegression(max_iter=2000, C=4.0, random_state=0)),
]

chay_bang("Bang B: TF-IDF ky tu", BANG_B, Xh_t, Xk_t, yh_t, yk_t)

Bang B: TF-IDF ky tu
Mo hinh                         Tap da hockiem dinh    Giay
KNN k=5                           0.8530      0.7779     1.2
cay quyet dinh                    0.9921      0.7278     7.1
rung 300 cay                      0.9922      0.8016    34.4
boosting                          0.9101      0.8063     7.6
hoi quy logistic                  0.8417      0.8070     0.3
